In [ ]:
import pandas as pd
import psycopg2
import DATABASE_CONFIG

conn = psycopg2.connect(
    dbname=DATABASE_CONFIG.DB_NAME,
    user=DATABASE_CONFIG.DB_USER,
    password=DATABASE_CONFIG.DB_PASSWORD,
    host=DATABASE_CONFIG.DB_HOST,
    port=DATABASE_CONFIG.DB_PORT
)

cursor = conn.cursor()

In [15]:
command = """
SELECT * FROM estado
"""
cursor.execute(command)
rows = cursor.fetchall()
id_estado = dict()
for row in rows:
    id_estado[row[1].upper()] = row[0]

print(id_estado)

{'ACRE': 1, 'AMAPÁ': 2, 'AMAZONAS': 3, 'MARANHÃO': 4, 'MATO GROSSO': 5, 'PARÁ': 6, 'RONDÔNIA': 7, 'RORAIMA': 8, 'TOCANTINS': 9}


In [16]:
df = pd.read_csv('../datasets/FocoQueimadasEstado.csv', sep=';')

df = df.rename(columns={
    'date': 'mes_ano',
    'class': 'tipo_area',
    'focuses': 'quantidade',
    'uf': 'id_estado',
})

print(df.columns)
df['mes_ano'] = df['mes_ano'].astype(str) + '/01'
df['id_estado'] = df['id_estado'].map(id_estado)

# for _, row in df.iterrows():
#     print(row)

data = list(df.itertuples(index=False, name=None))

Index(['mes_ano', 'tipo_area', 'quantidade', 'id_estado'], dtype='object')


In [17]:
from psycopg2.extras import execute_values
comando = """
    INSERT INTO relatorio_focos_queimadas
    (mes_ano, tipo_area, quantidade, id_estado)
    VALUES %s
"""
execute_values(cursor, comando, data)

conn.commit()


In [ ]:
conn.rollback()

In [18]:
cursor.close()
conn.close()